# Entregável 7 — Engenharia de Features

**Disciplina:** Aquisição de Biossinais  
**Equipe:** José Lessa & Matheus Rocha  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL (PhysioNet)  

---

## Objetivo

Otimizar e aprimorar o dataset matemático bruto do *Entregável 6* para torná-lo um terreno 
perfeito para os classificadores de Machine Learning (Validação Final). Isso envolve combinar 
features (conhecimento de domínio), escaloná-las (mesma unidade distributiva) e, 
imperativamente, **remover redundâncias brutais** e provar associação analítica de que as 
matemáticas levantadas até agora de fato servem para o Target Médico.

### Conteúdo abordado:
- Construção da Label Mestra (Target do Paciente: NORM vs Abnormal)
- Criação de Features Derivadas (Razões Clínicas)
- Standard Scaling (Z-score)
- Análise Multicolinearidade (Redundância por Correlação de Spearman)
- Validação Obrigatória: Teste Point-Biserial com Variável Resposta

## 1. Configuração e Imports

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configurações de plot
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Diretórios
BASE_DIR = Path('..')
DATA_DIR = Path('../../data/ptb-xl')
FIG_DIR = Path('figuras')
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')

## 2. Carregamento: Features (E.6) e Metadados Target (PTB-XL)

In [ ]:
# Tabela Bruta (Entregável 6)
csv_path = BASE_DIR / 'entregavel-6' / 'dataset_features_extraidas.csv'
df_feat = pd.read_csv(csv_path)

# Metadata (PTB-XL com os Ground Truth Diagnósticos Médicos)
df_meta = pd.read_csv(DATA_DIR / 'ptbxl_database.csv')

print(f"Dataset E.6 carregado: {df_feat.shape[0]} instâncias, {df_feat.shape[1]-3} features puras.")

### 2.1 Construção do Target (Rótulo de Resposta Médica)

O PTB-XL salva os rótulos numa coluna `scp_codes` da forma `{'NORM': 100.0, 'CLBBB': 15.0...}`.
Extrairemos uma classificação binária baseada na chave mágica **NORM (Normal)**. Se o dicionário aponta forte índice probabilístico de 'NORM', o paciente é *Saudável*. Caso contrário, é *Doente/Anormal* (MI, STTC, HYP etc).

In [ ]:
def extract_binary_target(scp_str):
    """Converte o dictionary textual do csv do ptbxl pra True se o ecg for Saudável, False se anómalo"""
    # parse string pra python dict
    scp_dict = ast.literal_eval(scp_str)
    
    # PTB-XL adota as mega-classes. NORM é Saudável.
    # Verifica se diagnosticou como NORM primario (Prob > 50)
    if 'NORM' in scp_dict and scp_dict['NORM'] > 50.0:
        return 1  # Classe Saudável (+)
    else:
        return 0  # Classe Anormal (-)

# Gerar mapa 
mapa_diagnosticos = {row['ecg_id']: extract_binary_target(row['scp_codes']) for index, row in df_meta.iterrows()}

# Mesclar no DataFrame de Features
df_feat['target_is_norm'] = df_feat['ecg_id'].map(mapa_diagnosticos)

print("Target inserido com sucesso.")
print("Distribuição nas instâncias janeladas:")
print(df_feat['target_is_norm'].value_counts().rename({0: 'Anormal (0)', 1: 'Saudável (1)'}))

## 3. Criação de Novas Features Derivadas

Segundo conhecimentos prévios cardíacos, certas razões e proporções indicam mais do que fatores matemáticos isolados.

In [ ]:
def engenhar_features(df):
    df_eng = df.copy()
    
    # 1. Razão LF/HF (Frequência)
    # Análogo ao balanço simpato-vagal na Análise HRV Clássica (variabilidade da freq cardíaca).
    # No ECG estrutural, indica rácio entre potências lentas (ondas base) por potência rápida (QRS arestas).
    df_eng['f_ratio_LF_HF'] = df_eng['f_power_LF'] / (df_eng['f_power_HF'] + 1e-6)
    
    # 2. Razão Hjorth (Tempo)
    # Relação Mobillity/Complexity. Uma onda muito íngreme e caótica tem uma relação diferente de uma senóide perfeita.
    df_eng['t_ratio_hjorth_mob_comp'] = df_eng['hjorth_mobility'] / (df_eng['hjorth_complexity'] + 1e-6)
    
    # 3. Wavelet Energy Ratio
    # Ratio entre Detalhes mais profundos (alta variação cD1) pela Aproximação Global (tendência cA4)
    df_eng['wt_ratio_D1_A4'] = df_eng['wt_energy_cD1'] / (df_eng['wt_energy_cA4'] + 1e-6)
    
    # 4. Agregações Temporais Derivadas (Tratar a Série Poincaré como Ratio)
    # Já feita em partes, era a 'nl_poincare_ratio' calculada por instinto prévio.
    
    return df_eng

df_eng = engenhar_features(df_feat)
print("3 Novas razões combinatórias integradas como features.")

## 4. Normalização (Escalonamento Estatístico)

O RMS tem amplitude em $mV$ e média `~0.1`. A Entropia espectral opera em limites `-4.0`.
As frequências chegam a `10.0`. Misturar isso num SVM ou numa Rede Neural arruína os 
gradientes do otimizador. O Standard Scaling é universalmente OBRIGATÓRIO por trás de algoritmos euclidianos/distânciais (k-NN, PCA).
$$Z = \frac{X - \mu}{\sigma}$$

In [ ]:
# Separar metadados não modificados
cols_meta = ['ecg_id', 'derivacao', 'janela_instancia', 'target_is_norm']
cols_features = [c for c in df_eng.columns if c not in cols_meta]

scaler = StandardScaler()

# Atenção: Normalizamos derivacao a derivacao para não misturar domínios da onda II longo vs V1 curta!
df_scaled_list = []
for derivacao in df_eng['derivacao'].unique():
    mask = df_eng['derivacao'] == derivacao
    subset_df = df_eng[mask].copy()
    
    # Fit-Transform nas numeric columns
    subset_df[cols_features] = scaler.fit_transform(subset_df[cols_features])
    df_scaled_list.append(subset_df)

df_scaled = pd.concat(df_scaled_list).sort_index()

print("StandardScaler aplicado Z-score separadamente em leads DII e V1.")
print("Exemplo (Média agora oscila ao redod de 0 e STD é 1):\n")
display(df_scaled[cols_features].head(3).round(4))

## 5. Análise e Poda de Redundância Colinear

Se o RMS e a Média Absoluta (MAV) flutuam 99% simetricamente na série de ECG, ambas carregam rigorosamente a **MESMA** informação patológica. Manter as duas inflaciona algoritmos. A redundância é inimiga da *Feature Selection* (E.9).

In [ ]:
# Analisar as Features Extraidas vs Features Extraidas
plt.figure(figsize=(16, 12))
corr_matrix_all = df_scaled[cols_features].corr(method='spearman')  # Spearman é robusto a nao linearidade 

# Triângulo superior da matriz para não mostrar espelhado
mask = np.triu(np.ones_like(corr_matrix_all, dtype=bool), k=0)

sns.heatmap(corr_matrix_all, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, cbar_kws={'shrink': .7})

plt.title('Mega Matriz de Redundância (Todas as 30 Features)', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.savefig(FIG_DIR / 'matriz_redundancia_todas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

In [ ]:
def remove_collinear_features(df, cols, threshold=0.95):
    """
    Deleta iterativamente uma feature quando sua correlação Absoluta de Spearman for maior que Threshold.
    """
    corr_matrix = df[cols].corr(method='spearman').abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # Encontra pares excessivamente redundantes (Ex. r = 0.999)
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    
    print(f"-> Encontradas {len(to_drop)} redundâncias gravíssimas (correlação > {threshold})")
    print("Features Dropadas: ", to_drop)
    return to_drop

# Limpar nosso df_scaled:
redundants = remove_collinear_features(df_scaled, cols_features, threshold=0.95)

# Subconjunto de Características Refinadas!
cols_purificadas = [c for c in cols_features if c not in redundants]

df_final = df_scaled[cols_meta + cols_purificadas].copy()

print(f"\nDimensão antes: {len(cols_features)} | Dimensão após poda: {len(cols_purificadas)}")

## 6. Validação Estatística: Correlação Variável Resposta (A Prova de Fogo)

Até o momento, inventamos features dezenas de vezes. Qual delas serve para a 
medicina de fato? 

Se a **Entropia de Shannon** tiver Correlação Biserial de Ponto alta (positiva/negativa) com nosso alvo binário (0-Anormal VS 1-Saudável), sabemos que a **desorganização do sinal importa p/ o dignóstico final**. Se der P-value > 0.05 e Ratio `~0`, essa feature é cegueira para esta tarefa clínica especifica.

In [ ]:
val_results = []
target_vector = df_final['target_is_norm'].values

for f in cols_purificadas:
    f_vector = df_final[f].values
    
    # Point-Biserial é o equivalente matematico da correlação de Pearson de um vetor Numérico Contínuo (z-score) versus Vetor Dicotômico Binary
    r_pb, p_val = stats.pointbiserialr(target_vector, f_vector)
    
    val_results.append({
        'Feature Ouro': f,
        'Point-Biserial (R)': r_pb,
        'Impacto Absoluto (|r|)': abs(r_pb),
        'p-value': p_val,
        'Relevância Clínica Médica?': '✅ Significativa' if p_val < 0.05 else '❌ Descartável'
    })

df_val_final = pd.DataFrame(val_results).sort_values('Impacto Absoluto (|r|)', ascending=False)

print('=' * 90)
print("VALIDAÇÃO: TOP 15 MELHORES PREDITORES DIRETOS DO DIAGNÓSTICO DO PACIENTE")
print('=' * 90)
display(df_val_final.head(15).round(4))

In [ ]:
# Plotando os Taffarelli top 2 para visualizar a separabilidade intuitivamente
f1 = df_val_final.iloc[0]['Feature Ouro']
f2 = df_val_final.iloc[1]['Feature Ouro']

plt.figure(figsize=(10, 6))
sns.violinplot(data=df_final, x='target_is_norm', y=f1, palette=['crimson', 'mediumseagreen'])
plt.title(f'Visualização da Top 1 Feature: [{f1}] separando Anormais (0) vs Saudáveis (1)', fontweight='bold')
plt.xlabel('Diagnóstico Real Médico (0=Anormal, 1=Saudável)')
plt.ylabel(f'Ocorrência Z-Score ({f1})')

plt.tight_layout()
plt.savefig(FIG_DIR / 'violin_top_feature.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva.')

## 7. Salvamentos Finais do Entregável
Exportamos os relatórios validando e o dataset podado pro proximo algoritmo do fluxo de Classificação Inteligente.

In [ ]:
output_csv = FIG_DIR.parent / 'dataset_features_engenheirado.csv'
df_final.to_csv(output_csv, index=False)

print(f"-> Arquivo Mestre exportado livre de ruídos/colineares para PCA: {output_csv}")

output_csv_val = FIG_DIR.parent / 'tabela_validacao_pointbiserial.csv'
df_val_final.to_csv(output_csv_val, index=False)
print(f"-> Planilha de Justificativas Cientificas (Target PB-R): {output_csv_val}")